NHANES Dataset
==============

Research Question:

Is food insecurity associated with signs of inflammation or other chronic diseases?


Welcome to the CAIPH datathon.

Your dataset is the long term NHANES data set. If you aren't familiar with the data set, the wikipedia has a good summary:

> The National Health and Nutrition Examination Survey (NHANES) is a survey research program conducted by the National Center for Health Statistics (NCHS) to assess the health and nutritional status of adults and children in the United States, and to track changes over time.[1] The survey combines interviews, physical examinations and laboratory tests.[citation needed]
>The NHANES interview includes demographic, socioeconomic, dietary, and health-related questions. The examination component consists of medical, dental, and physiological measurements, as well as laboratory tests administered by medical personnel.[citation needed]
>The National Health Survey Act was passed in 1956. This allowed legislative authorization to provide current statistical data on the amount, distribution, and effects on illness and disability in the United States.[2]
>

Unlike some other data sets in this datathon, where datathoners will load their data via CSV, NHanes big and complicated enough that Christopher J. Endres has created an R library explicitly for the purpose of search, loading, and understanding this data set. The R vignette for this library is included in full below, with minor modification to make it interactive. Any errors or issues in this translatio are the fault of the Datathon team.

Before You Start
================

It is useful to have some idea of what you might want to look at. A good first question would be trying to extract data similar to the stroke data set that another team [may be tackling](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset) at the datathon. Most of the demographic data collected in that data set is available in NHANES - could a model trained on one set be applied to another? We can find stroke related data set in the NHANES data like this:

In [3]:
ensure_library <- function(pkg) {
  pkg_name <- deparse(substitute(pkg))
  if (!requireNamespace(pkg_name, quietly = TRUE)) {
    install.packages(pkg_name)
  }
  library(pkg_name, character.only = TRUE)
}
library(tidyverse)
ensure_library(nhanesA)
nhanesSearch("food security") %>% head()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


,Variable.Name,Variable.Description,Data.File.Name,Data.File.Description,Begin.Year,EndYear,Component,Use.Constraints
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ADfdsec,Adult food security category for last 12 months,FSQ,Food Security,1999,2000,Questionnaire,None
2,CHfdsec,Child food security category for last 12 months,FSQ,Food Security,1999,2000,Questionnaire,None
3,HHfdsec,Household food security category for last 12 months,FSQ,Food Security,1999,2000,Questionnaire,None
4,FSDAD,Adult food security category for last 12 months,FSQ_E,Food Security,2007,2008,Questionnaire,None
5,FSDCH,Child food security category for last 12 months,FSQ_E,Food Security,2007,2008,Questionnaire,None
6,FSDHH,Household food security category for last 12 months,FSQ_E,Food Security,2007,2008,Questionnaire,None


# Introducing nhanesA 

#### Christopher J. Endres 

#### 2025-10-13 

## Background

The nhanesA R package was developed to allow investigators to easily
explore and retrieve data from the National Health and Nutrition
Examination Survey (NHANES). The survey assesses overall health and
nutrition of adults and children in the United States, and is conducted
by the National Center for Health Statistics (NCHS). NHANES data are
publicly available at: <https://www.cdc.gov/nchs/nhanes/> and are
reported in thousands of peer-reviewed journal publications every year.

## NHANES Data

Since 1999, the NHANES survey has been conducted continuously, and the
surveys during that period are referred to as "continuous NHANES" to
distinguish from several prior surveys. Continuous NHANES surveys are
grouped in two-year intervals, with the first interval being 1999-2000.

Most NHANES data are in the form of tables in SAS 'XPT' format. The
survey is grouped into five data categories that are publicly available,
as well as an additional category (Limited access data) that requires
written justification and prior approval before access. Package nhanesA
is intended mostly for use with the publicly available data, but some
information pertaining to the limited access data can also be retrieved.

The five publicly available data categories are: - Demographics (DEMO) -
Dietary (DIET) - Examination (EXAM) - Laboratory (LAB) - Questionnaire
(Q). The abbreviated forms in parentheses may be substituted for the
long form in nhanesA commands.

For limited access data, the available tables and variable names can be
listed, but the data cannot be downloaded directly. To indicate limited
access data in nhanesA functions, use: - Limited (LTD)

### List NHANES Tables

To quickly get familiar with NHANES data, it is helpful to display a
listing of tables. Use nhanesTables to get information on tables that
are available for a given category for a given year.

In [1]:
ensure_library(nhanesA)
nhanesTables('EXAM', 2005)


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



Data.File.Name,Data.File.Description
<chr>,<chr>
BPX_D,Blood Pressure
BMX_D,Body Measures
AUX_D,Audiometry
AUXTYM_D,Audiometry - Tympanometry
DXXFEM_D,Dual Energy X-ray Absorptiometry - Femur
OPXFDT_D,Ophthalmology - Frequency Doubling Technology
OHX_D,Oral Health
PAXRAW_D,Physical Activity Monitor
VIX_D,Vision


Note that the two-year survey intervals begin with the odd year. For
convenience, only a single 4-digit year is entered such that
`nhanesTables('EXAM', 2005)` and `nhanesTables('EXAM', 2006)` yield
identical output.

### List Variables in an NHANES Table

After viewing the output, we decide we are interested in table 'BMX_D'
that contains body measures data. To better determine if that table is
of interest, we can display detailed information on the table contents
using nhanesTableVars.

In [ ]:
nhanesTableVars('EXAM', 'BMX_D')

We see that there are 27 columns in table BMX_D. SEQN is a subject
identifier that is used to join information across tables.

### Import NHANES Tables

We now import BMX_D along with the demographics table DEMO_D.

In [ ]:
bmx_d  <- nhanes('BMX_D')
demo_d <- nhanes('DEMO_D')

We merge the tables and display several variables. Note that RIAGENDR,
like most categorical variables, is a coded field. By default, the
original coded values (1,2) are translated to (Male, Female).

In [ ]:
bmx_demo <- merge(demo_d, bmx_d)
options(digits=4)
select_cols <- c('RIAGENDR', 'BMXHT', 'BMXWT', 'BMXLEG', 'BMXCALF', 'BMXTHICR')
print(bmx_demo[5:8,select_cols], row.names=FALSE)

### Display Codebook

For each variable, NHANES provides a codebook, which is a basic
description of the variable and also includes the distribution or range
of values. We can use nhanesCodebook to list the codebook definition for
the gender field RIAGENDR in table DEMO_D.

In [ ]:
nhanesCodebook('DEMO_D', 'RIAGENDR')

### Apply Code Translations

As a default, the nhanes function will translate coded values. To ensure
proper interpretation of variables, it is recommended to always use
`nhanes` with the default option of translate = TRUE. However, you may
also customize the translation of coded fields manually using
nhanesTranslate. Customized translation of coded fields is a three step
process. 1: Download the table using nhanes with translate = FALSE 2:
Select the table variables to translate 3: Pass the table and variable
list to nhanesTranslate

In [ ]:
bpx_d <- nhanes('BPX_D', translate=FALSE)
head(bpx_d[,6:11])

In [ ]:
bpx_d_vars  <- nhanesTableVars('EXAM', 'BPX_D', namesonly=TRUE)
#Alternatively may use bpx_d_vars = names(bpx_d)
bpx_d <- nhanesTranslate('BPX_D', bpx_d_vars, data=bpx_d)

In [ ]:
head(bpx_d[,6:11])

Some discretion should be applied when translating coded columns as code
translations can be quite long. To improve readability the translation
string is restricted to a default length of 128 but can be set as high
as 1024.

### Download a Complete Survey

The primary goal of nhanesA is to enable fully customizable processing
of select NHANES tables. However, it is quite easy to download entire
surveys using nhanesA functions. Say we want to download every
questionnaire in the 2007-2008 survey. We first get a list of the table
names by using nhanesTables with namesonly = TRUE. The tables can then
be downloaded using nhanes with lapply.

In [ ]:
q2007names  <- nhanesTables('Q', 2007, namesonly=TRUE)
q2007tables <- lapply(q2007names, nhanes)
names(q2007tables) <- q2007names

## Special Cases

Some NHANES measurements require special handling, e.g. due to
statistical considerations. Furthermore, there are surveys conducted
outside the scope of the continuous survey, but that are provided in a
similar format such that nhanesA can be readily adapted to retrieve
their data. Note that nhanesA cannot be used to handle accelerometer
data from 2003-2006. For those data, please see package
[accelerometry](https://cran.r-project.org/package=accelerometry).

### Pre-pandemic data

Due to COVID, data collection for the 2019-2020 cycle was not completed.
In order to make better use of data that were collected, data from the
2017-2018 survey were included to form a representative sample of
2017-March 2020 pre-pandemic data. The table structure and variable
format of pre-pandemic tables is essentially identical to standard
continuous NHANES. The table names include the prefix "P\_". To list
pre-pandemic tables, use 'P' or 'p' as the year.

In [ ]:
#List all pre-pandemic tables
nhanesSearchTableNames('^P_')
#List table variables
nhanesTableVars('EXAM', 'P_AUX', namesonly=TRUE)
#List pre-pandemic EXAM tables
nhanesTables('EXAM', 'P')
#Table import, variable translation, and codebook display operate as usual
p_dxxfem <- nhanes('P_DXXFEM')
nhanesTranslate('P_BMX', 'BMDSTATS')
nhanesCodebook('P_INS', 'LBDINSI')

### Dual Energy X-Ray Absorptiometry

Dual Energy X-Ray Absorptiometry (DXA) data were acquired from
1999-2006. These data were found to contain a higher amount of missing
values than usual, and a multiple imputation process was applied to fill
in missing and invalid values. More information may be found at
<https://wwwn.cdc.gov/nchs/nhanes/dxa/dxa.aspx>. By default the DXA data
are imported into the R environment, however, because the tables are
quite large it may be desirable to save the data to a local file then
import to R as needed. When nhanesTranslate is applied to DXA data, only
the 2005-2006 translation tables are used as those are the only DXA
codes that are currently available in html format.

In [ ]:
#Import into R
dxx_b <- nhanesDXA(2001)
#Save to file
nhanesDXA(2001, destfile="dxx_b.xpt")
#Import supplemental data
dxx_c_s <- nhanesDXA(2003, suppl=TRUE)
#Apply code translations
dxalist <- c('DXAEXSTS', 'DXIHE')
dxx_b <- nhanesTranslate("dxxb",colnames=dxalist, data=dxx_b, dxa=TRUE)

### NHANES National Youth Fitness Survey (NNYFS)

NNYFS is a 2012 ancillary study conducted to assess physical activity
and fitness levels of children and teens from ages 3-15. This study
consists of questionnaires and basic measurements. There are no LAB
tables. NNYFS table names include the prefix "Y\_". To list tables, use
'Y' or 'y' as the year.

In [ ]:
#List NNYFS EXAM tables
nhanesTables('EXAM', 'Y')
#Table import and variable translation operate as usual
y_cvx <- nhanes('Y_CVX')
nhanesTranslate('Y_CVX','CVXPARC')

## Searching for tables and variables

The NHANES repository is extensive, thus it is helpful to perform a
targeted search to identify relevant tables and variables. There are
several nhanesA functions that allow the user to search using different
criteria including the variable description, variable name, and table
name pattern.

### Searching across the comprehensive list of NHANES variables

Comprehensive lists of NHANES variables are maintained for each data
group. For example, the demographics variables are available at
<https://wwwn.cdc.gov/nchs/nhanes/search/variablelist.aspx?Component=Demographics>.
The nhanesSearch function allows the investigator to input search terms,
match against the comprehensive variable descriptions, and retrieve the
list of matching variables. Matching search terms (variable descriptions
must contain one of the terms) and exclusive search terms (variable
descriptions must NOT contain any exclusive terms) may be provided. The
search can be restricted to a specific survey range as well as specific
data groups.

In [ ]:
# nhanesSearch use examples
#
# Search on the word bladder, restrict to the 2001-2008 surveys, 
# print out 50 characters of the variable description
nhanesSearch("bladder", ystart=2001, ystop=2008, nchar=50)
#
# Search on "urin" (will match urine, urinary, etc), from 1999-2010, return table names only
nhanesSearch("urin", ignore.case=TRUE, ystop=2010, namesonly=TRUE)
#
# Search on "urin", exclude "During", search surveys from 1999-2010, return table names only
nhanesSearch("urin", exclude_terms="during", ignore.case=TRUE, ystop=2010, namesonly=TRUE)
#
# Restrict search to 'EXAM' and 'LAB' data groups. Explicitly list matching and exclude terms, leave ignore.case set to default value of FALSE. Search surveys from 2009 to present.
nhanesSearch(c("urin", "Urin"), exclude_terms=c("During", "eaten during", "do during"), data_group=c('EXAM', 'LAB'), ystart=2009)
#
# Search on "tooth" or "teeth", all years
nhanesSearch(c("tooth", "teeth"), ignore.case=TRUE)
#
# Search for variables where the variable description begins with "Tooth"
nhanesSearch("^Tooth")

### Searching for tables that contain a specific variable

nhanesSearch is a versatile search function as it imports the
comprehensive variable lists to a data frame. That allows for detailed
conditional extraction of the variables. However, each call to
nhanesSearch takes up to a minute or more to process. Faster processing
can be achieved when we know the name of a specific variable of interest
and we look only for exact matches to the variable name. Function
nhanesSearchVarName matches a given variable name in the html directly,
then only the matching elements are converted to a data frame.
Consequently, a call to nhanesSearchVarName executes much faster than
nhanesSearch; typically under 30s. nhanesSearchVarName is useful for
finding all data tables that contain a given variable.

In [ ]:
#nhanesSearchVarName use examples
nhanesSearchVarName('BPXPULS')

```
nhanesSearchVarName('CSQ260i', includerdc=TRUE, nchar=38, namesonly=FALSE)
```{R}

    ##   Variable.Name                   Variable.Description Data.File.Name
    ## 1       CSQ260i Do you now have any of the following p        CSX_G_R
    ## 2       CSQ260i Do you now have any of the following p          CSX_H
    ##   Data.File.Description Begin.Year EndYear   Component Use.Constraints
    ## 1         Taste & Smell       2012    2012 Examination        RDC Only
    ## 2         Taste & Smell       2013    2014 Examination            None

### Searching for tables by name pattern

In order to group data across surveys, it is useful to list all
available tables that follow a given naming pattern. Function
nhanesSearchTableNames is used for such pattern matching. For example,
if we want to work with all available body measures data we can retrieve
the full list of available tables with nhanesSearchTableNames('BMX').
The search is conducted over the comprehensive table list, which is much
smaller than the comprehensive variable list, such that a call to
nhanesSearchTableNames takes only a few seconds.

```{R}
# nhanesSearchTableNames use examples
nhanesSearchTableNames('BMX')
```


In [ ]:
nhanesSearchTableNames('HPVS', includerdc=TRUE, nchar=42, details=TRUE)

#### Please send any feedback or requests to <cjendres1@gmail.com>. Hope you enjoy your experience with nhanesA!

Sincerely,
Christopher J. Endres